## Introduction

This notebook is a debugging notebook for the low level Jupyter Comms communications within [CubeVis](https://github.com/casangi/cubevis). The motivation for the ``cubevis`` shift from using only [websockets](https://pypi.org/project/websockets/) to supporting ``websockets``, ``Jupyter Comms`` and ``Colab Comms`` was the desire to support Colab.

Initially, it seemed as though the transition would be relatively straight forward for the original ``websockets`` based implementation. Colab has integrated proxy support:
```
from google.colab.output import eval_js
port = 8888
proxy_url = eval_js(f"google.colab.kernel.proxyPort({port})")
```
However, Google's proxy server does not support full protocol upgrades. In particular, it is not possible to upgrade a proxies port to support the websocket protocol. This truth only came to light after significant effort. After the necessary stages of grief, the decision was made to rework ``cubevis`` communications to support ``websockets``, ``Jupyter Comms`` and ``Colab Comms``, and because this would be a significant investment, we decided to also introduce multiplexed communications for better communication support, both for current commuications as well as future remote execution.

First ``websockets`` support was restored. This proved to be relatively straight forward. The only complication was the introduction of multiplexed communications to replace the original use of a dedicated websocket for each type of communication. However, this allow for development and testing of a code organization which allowed for the specifics of the channels, whether comms based or websockets based, to be isolated behind an abstract base class.

Next support for ``Jupyter Comms`` was introduced. This worksheet served as the development and debugging platform for this flavor of low level communications channel. We started with the most recent version of Jupyter Lab (March 2026) which was first introduced on May 15, 2023. This is probably the most secure version of Jupyter notebooks currently. This led to difficulties.

## Debugging setup with github branch/tag selection

In [ ]:
!pip install casatasks bokeh==3.9 scipy regions

In [1]:
import os
tag = 'jupyter-debug-0021'
os.environ['CUBEVIS_JS_TAG'] = tag
os.environ['CUBEVIS_DEBUG'] = '1'
os.makedirs(os.path.expanduser('~/.casa/data'),exist_ok=True)

In [ ]:
!pip install git+https://github.com/casangi/cubevis.git@{tag}

## Import ColabCommsTransport

In [ ]:
import asyncio
from bokeh.io import show, output_notebook
from cubevis.bokeh.transport._low_level_transport import ColabCommsTransport

## Test Jupyter Comms communications
This initializes the test and sets up a Python message handler for messages from JavaScript to Python.

### Jupyter Lab test

In [ ]:
os.environ['CUBEVIS_DEBUG'] = '1'
output_notebook()

import ipywidgets as widgets
from IPython.display import display

# Create a dedicated, scrollable output area with a border
debug_view = widgets.Output(layout={'border': '1px solid #4CAF50', 'height': '200px', 'overflow_y': 'scroll'})
display(debug_view)

transport = ColabCommsTransport(comm_mgr_id="isolation_test_pipe")
def py_receiver(msg):
    # This ensures the output is directed to the widget, not the void
    with debug_view:
        print(f"✅ Python Received: {msg}")

transport.set_message_callback(py_receiver)

# 1. Start the connection in the background
# This allows the cell to FINISH so the widget can render
import asyncio
asyncio.create_task(transport.connect())

### Colab test

#### Transport creation

In [ ]:
os.environ['CUBEVIS_DEBUG'] = '1'
output_notebook()

import ipywidgets as widgets
from IPython.display import display

# Create a dedicated, scrollable output area with a border
debug_view = widgets.Output(layout={'border': '1px solid #4CAF50', 'height': '200px', 'overflow_y': 'scroll'})
display(debug_view)

with debug_view:
    print("✅ Debug view created...")

def py_receiver_with_echo(msg):
    # Log to the scrollable widget
    with debug_view:
        print(f"📩 KERNEL RECEIVED: {msg}")
    
    if msg.get('type') == 'echo_request':
        # IMPORTANT: We use the 'transport._comm' which was just 
        # updated by the most recent JS connection.
        if transport._comm:
            transport._comm.send({
                "type": "echo_response",
                "content": f"ACK: {msg.get('content')}",
                "ts": msg.get('ts')
            })
            with debug_view:
                print("📤 KERNEL SENT ECHO")

transport = ColabCommsTransport(comm_mgr_id="isolation_test_pipe")
transport.set_message_callback(py_receiver_with_echo)

# This must be the LAST line and must not block.
await transport.connect()

#### JavaScript to Python

In [ ]:
from IPython.display import Javascript

display(Javascript(f'''
(async () => {{
    const target = "isolation_test_pipe";
    
    try {{
        console.log("JS: Opening channel...");
        const channel = await google.colab.kernel.comms.open(target, {{}});
        
        // 1. Setup the listener loop IMMEDIATELY
        // This must stay running to catch the response
        const listenerPromise = (async () => {{
            for await (const message of channel.messages) {{
                console.log("📥 JS RECEIVED FROM KERNEL:", message.data);
                
                const div = document.createElement('div');
                div.style.cssText = "border-left: 4px solid #4CAF50; padding: 10px; margin: 10px 0; background: #e8f5e9; font-family: monospace;";
                div.innerHTML = `<b>✅ SUCCESS: Bidirectional Link Active</b><br>Data: ${{JSON.stringify(message.data)}}`;
                document.body.appendChild(div);
                return; // Exit after first success for this test
            }}
        }})();

        // 2. Send the echo request
        console.log("JS: Sending request...");
        channel.send({{ 
            "type": "echo_request", 
            "content": "Hello World", 
            "ts": Date.now() 
        }});

        // 3. Keep the cell "active" for a few seconds to allow the loop to work
        await Promise.race([
            listenerPromise,
            new Promise(resolve => setTimeout(resolve, 5000))
        ]);

    }} catch (e) {{
        console.error("JS Error:", e);
    }}
}})();
'''))

#### Python to JavaScript

In [ ]:
# Cell: Python to JS Test
if transport.is_connected():
    print("📤 Sending message to JavaScript...")
    asyncio.create_task(transport.send_message({
        "type": "python_push",
        "content": "Hello from the Python Kernel!",
        "timestamp": 123456789
    }))
else:
    print("❌ Transport not connected. Run the JS handshake cell first.")

### Check connection

In [ ]:
if transport.is_connected():
    print("✅ Connection verified!")
else:
    # Give it another 2 seconds if you just ran Cell 1
    import asyncio
    await asyncio.sleep(2)
    print(f"Final Connection Status: {transport.is_connected()}")

### JavaScipt → Python
This sets up a message handler in JavaScript and also sends a message from JavaScript to Python

#### Colab test

In [ ]:
from IPython.display import Javascript

# Use the same ID you used in your Python transport setup
TEST_COMM_ID = "isolation_test_pipe"

display(Javascript(f'''
(async () => {{
    const target = "{TEST_COMM_ID}";
    console.log("Attempting manual raw handshake to:", target);
    
    try {{
        // 1. Directly use the Colab kernel API to open a channel
        // This is exactly what your class does internally.
        const channel = await google.colab.kernel.comms.open(target, {{}});
        
        // 2. Send a raw message immediately
        channel.send({{
            "type": "manual_test",
            "content": "Hello from a raw JS snippet!",
            "timestamp": Date.now()
        }});
        
        console.log("✅ Raw message sent successfully.");
    }} catch (e) {{
        console.error("❌ Failed to open raw channel. Is the Python listener running?", e);
    }}
}})();
'''))

#### Original

In [ ]:
%%javascript
(function verify_bidirectional() {
    const pipe_id = "isolation_test_pipe";
    const comm = window["comm_" + pipe_id];

    if (comm) {
        // 1. ATTACH THE LISTENER
        // This is the part that was likely missing or not active
        comm.onMsg = (msg) => {
            const data = msg.content.data;
            console.log("📢 JS RECEIVED FROM PYTHON:", data);
            
            // Also display it in the notebook for visual proof
            const display = document.createElement("div");
            display.style.padding = "10px";
            display.style.marginTop = "10px";
            display.style.background = "#eef";
            display.style.border = "1px solid #2196f3";
            display.innerHTML = `✅ <b>Success!</b> Received: ${JSON.stringify(data)}`;
            element.append(display);
        };

        // 2. TRIGGER THE TEST (JS -> PY)
        comm.send({ text: "Checking JS -> PY path..." });
        element.append("📡 JS -> PY Message Sent. Now run the Python 'You win' cell.");
    } else {
        element.append("❌ Bridge not found. Ensure Cell 1 is blue.");
    }
})();

### Python → JavaScript

In [ ]:
await transport.send_message({"type": "FINISH_TEST", "content": "You win!"})